
# U-Net Architecture Overview

Обучим собственную U-Net для задачи Image Colorization.

U-Net — это свёрточная архитектура типа **encoder-decoder**, первоначально предложенная для сегментации изображений. Она состоит из двух основных частей: encoder, который постепенно уменьшает пространственное разрешение и извлекает признаки, и decoder, который восстанавливает разрешение для получения попиксельных предсказаний.

**Почему U-Net подходит для задачи колоризации?**

Это архитектура, которая способна преобразовывать одно пространственное представление изображения в другое.

В задаче сегментации мы передаём на вход RGB-изображение и получаем на выходе предсказания для каждого пикселя. При многоклассовой сегментации выход модели имеет форму `(B, C, H, W)`, где `C` — количество классов. С помощью `softmax` можно получить вероятности классов, а `argmax` позволяет выбрать класс для каждого пикселя и сформировать двумерную маску `(H, W)`.

В задаче колоризации мы передаём чёрно-белое изображение и хотим получить на выходе два канала с предсказанными значениями `a` и `b`. Получается, что обе задачи требуют построить новое изображение, сохранив пространственное соответствие с исходным.

Допустим, у нас есть конкретный пиксель со значением `L = 50`. Можем ли мы только по этому числу определить, какого цвета должен быть пиксель? Нет! Одному значению lightness может соответствовать множество разных цветов.

Поэтому модель должна учитывать не только значение конкретного пикселя, но и контекст вокруг него: текстуры, границы, расположение объектов и общую структуру изображения.

$$
f(L) = \widehat{ab}
$$

Здесь `L` — вся карта lightness изображения, а $\widehat{ab}$ — предсказанные цветовые каналы.

**Особенность U-Net — skip connections.** Они соединяют соответствующие уровни encoder и decoder. Промежуточные feature maps из encoder передаются напрямую в decoder и объединяются с его признаками через конкатенацию по каналам. Это позволяет использовать одновременно высокоуровневый контекст и пространственные детали изображения, которые могли потеряться при уменьшении разрешения.

В нашей задаче модель будет получать нормализованный канал `L` (lightness) и предсказывать два цветовых канала `ab`. Таким образом, в отличие от многоклассовой сегментации, мы решаем **задачу регрессии**, в которой для каждого пикселя необходимо предсказать непрерывные значения цвета.

**Размерности входа и выхода:**

- Input: `(B, 1, 128, 128)` — нормализованный L-канал (lightness).
- Output: `(B, 2, 128, 128)` — предсказанные каналы ab.


In [1]:
import numpy as np 
import matplotlib.pyplot as plt
import sys
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
from torch import nn
import os 
from image_colorization.color_utils import denormalize_lab, prepare_lab
from image_colorization.unet import DoubleConv, Down, Up, UNet
from image_colorization.dataset import ColorizationDataset

In [2]:
images_dir = Path('../VOCdevkit/VOC2012/JPEGImages')
train_path = Path('../data/splits/train.txt')
val_path = Path('../data/splits/val.txt')

train_transform = transforms.Compose([
    transforms.Resize(128),
    transforms.CenterCrop(128)
]) 
train_dataset = ColorizationDataset(images_dir=images_dir,
                                    split_path = train_path,
                                    transform = train_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)

val_transform = transforms.Compose([
    transforms.Resize(128),
    transforms.CenterCrop(128)
])
val_dataset = ColorizationDataset(images_dir=images_dir,
                                  split_path = val_path,
                                  transform = val_transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [3]:
batch = next(iter(train_loader))

In [4]:
L, ab = batch
print(L.shape)
print(ab.shape)

torch.Size([32, 1, 128, 128])
torch.Size([32, 2, 128, 128])


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [6]:
from tqdm.auto import tqdm
from IPython.display import display

def training_epoch(model, optimizer, criterion, train_loader, device):
    total_loss = 0.0
    total_samples = 0

    model.train()
    for L, ab in train_loader:
        L = L.to(device)
        ab = ab.to(device)

        optimizer.zero_grad()
        pred = model(L)
        loss = criterion(pred, ab)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * L.size(0)
        total_samples += L.size(0)

    return total_loss / total_samples


def validation_epoch(model, criterion, val_loader, device):
    total_loss = 0.0
    total_samples = 0
    model.eval()

    with torch.no_grad():
        for L, ab in val_loader:
            L = L.to(device)
            ab = ab.to(device)

            pred = model(L)
            loss = criterion(pred, ab)
            total_loss += loss.item() * L.size(0)
            total_samples += L.size(0)

    return total_loss / total_samples

In [7]:
def train(model, criterion, optimizer, train_loader, val_loader, num_epoches, device):
    train_loss = []
    val_loss = []

    plot_handle = None
    pbar = tqdm(range(1, num_epoches + 1), desc="Training")

    for epoch in pbar:

        train_loss.append(training_epoch(model, optimizer,
                                         criterion, train_loader, device))

        val_loss.append(validation_epoch(model, criterion,
                                         val_loader, device))

        pbar.set_postfix(
            train_loss=f"{train_loss[-1]:.4f}",
            val_loss=f"{val_loss[-1]:.4f}"
        )

        fig, ax = plt.subplots(figsize=(8, 4))

        ax.plot(range(1, epoch + 1), train_loss,
                marker="o", label="Train loss")

        ax.plot(range(1, epoch + 1), val_loss,
                marker="o", label="Validation loss")

        ax.set_xlabel("Epoch")
        ax.set_ylabel("L1 Loss")
        ax.legend()
        ax.grid(alpha=0.3)

        if plot_handle is None:
            plot_handle = display(fig, display_id=True)
        else:
            plot_handle.update(fig)

        plt.close(fig)

    return train_loss, val_loss